In [1]:
# Cell 1: Notebook purpose
print("TMG-GAN Colab Runner (Git-first workflow)")
print("Order: local edit/push -> Colab pull -> preflight -> train -> evaluate")

TMG-GAN Colab Runner (Git-first workflow)
Order: local edit/push -> Colab pull -> preflight -> train -> evaluate


In [2]:
# Cell 2: Runtime and run metadata
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import torch

PY = sys.executable
DATASET = os.environ.get("TMG_DATASET", "CICIDS2017")
RUN_STEM = os.environ.get("TMG_RUN_STEM", "tmg_gan_tabular")
RUN_TAG = os.environ.get("TMG_RUN_TAG", "").strip()
RUN_NAME_OVERRIDE = os.environ.get("TMG_RUN_NAME", "").strip()
RESUME_MODE = os.environ.get("TMG_RESUME", "0") == "1"
BASELINE_RUN_NAME = os.environ.get("TMG_BASELINE_RUN_NAME", RUN_STEM).strip() or RUN_STEM

REPO_URL = os.environ.get("TMG_REPO_URL", "https://github.com/Omayer111/TMG-GAN.git")
REPO_ROOT = Path(os.environ.get("TMG_REPO_ROOT", "/content/TMG-GAN"))
BRANCH = os.environ.get("TMG_BRANCH", "main")
ALLOW_DIRTY = os.environ.get("TMG_ALLOW_DIRTY", "0") == "1"

if RUN_NAME_OVERRIDE:
    RUN_NAME = RUN_NAME_OVERRIDE
elif RUN_TAG:
    RUN_NAME = f"{RUN_STEM}_{RUN_TAG}"
else:
    RUN_NAME = f"{RUN_STEM}_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"

print(f"Python: {PY}")
print(f"Dataset: {DATASET}")
print(f"Run mode: {'RESUME' if RESUME_MODE else 'FRESH'}")
print(f"Run name: {RUN_NAME}")
print(f"Repo root: {REPO_ROOT}")
print(f"Branch: {BRANCH}")

Python: /usr/bin/python3
Dataset: CICIDS2017
Run mode: FRESH
Run name: tmg_gan_tabular_20260310_094614
Repo root: /content/TMG-GAN
Branch: main


In [3]:
# Cell 3: Git sync from GitHub (single source of truth for code)
def run(cmd, cwd=None, check=True):
    print("+", " ".join(cmd))
    p = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        text=True,
        capture_output=True,
    )
    if p.stdout:
        print(p.stdout.rstrip())
    if p.stderr:
        print(p.stderr.rstrip())
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(cmd)}")
    return p

if not REPO_ROOT.exists():
    run(["git", "clone", REPO_URL, str(REPO_ROOT)])

run(["git", "-C", str(REPO_ROOT), "remote", "set-url", "origin", REPO_URL], check=False)
run(["git", "-C", str(REPO_ROOT), "fetch", "origin", "--prune"])

if BRANCH == "auto":
    head_ref = run(
        ["git", "-C", str(REPO_ROOT), "symbolic-ref", "refs/remotes/origin/HEAD"],
        check=False,
    ).stdout.strip()
    BRANCH = head_ref.split("/")[-1] if head_ref else "main"

status = run(["git", "-C", str(REPO_ROOT), "status", "--porcelain"], check=False).stdout.strip()
if status and not ALLOW_DIRTY:
    raise RuntimeError(
        "Runtime repo has local edits. Commit/stash/discard first, or set TMG_ALLOW_DIRTY=1 intentionally."
    )

run(["git", "-C", str(REPO_ROOT), "checkout", BRANCH])
run(["git", "-C", str(REPO_ROOT), "pull", "--rebase", "origin", BRANCH])

REPO_COMMIT = run(["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"]).stdout.strip()
print("Synced commit:", REPO_COMMIT)

os.environ["TMG_RESULTS_ROOT"] = str(REPO_ROOT / "results")
print("TMG_RESULTS_ROOT =", os.environ["TMG_RESULTS_ROOT"])

+ git clone https://github.com/Omayer111/TMG-GAN.git /content/TMG-GAN
Cloning into '/content/TMG-GAN'...
+ git -C /content/TMG-GAN remote set-url origin https://github.com/Omayer111/TMG-GAN.git
+ git -C /content/TMG-GAN fetch origin --prune
+ git -C /content/TMG-GAN status --porcelain
+ git -C /content/TMG-GAN checkout main
Your branch is up to date with 'origin/main'.
Already on 'main'
+ git -C /content/TMG-GAN pull --rebase origin main
Already up to date.
From https://github.com/Omayer111/TMG-GAN
 * branch            main       -> FETCH_HEAD
+ git -C /content/TMG-GAN rev-parse --short HEAD
3217940
Synced commit: 3217940
TMG_RESULTS_ROOT = /content/TMG-GAN/results


In [ ]:
# Cell 4: Resolve project paths and inspect trainer capabilities
RESULTS_ROOT = Path(os.environ["TMG_RESULTS_ROOT"]).resolve()
PROJECT_ROOT = RESULTS_ROOT / "thesis_tmg_pipeline"
DATA_ROOT = RESULTS_ROOT / "datasets"
DATASET_DIR = DATA_ROOT / DATASET

TRAIN_SCRIPT = PROJECT_ROOT / "scripts" / "train_tmg_gan.py"
PREFLIGHT_SCRIPT = PROJECT_ROOT / "scripts" / "colab_preflight.py"
VISUALIZE_SCRIPT = PROJECT_ROOT / "scripts" / "visualize_run_metrics.py"
REQUIREMENTS_FILE = PROJECT_ROOT / "requirements.txt"

for p in (RESULTS_ROOT, PROJECT_ROOT, TRAIN_SCRIPT, PREFLIGHT_SCRIPT, VISUALIZE_SCRIPT, REQUIREMENTS_FILE):
    if not p.exists():
        raise FileNotFoundError(f"Required path not found: {p}")

help_text = run([PY, str(TRAIN_SCRIPT), "-h"], check=False).stdout
SUPPORTED_FLAGS = set()
for line in help_text.splitlines():
    if line.strip().startswith("--"):
        SUPPORTED_FLAGS.add(line.strip().split()[0])

print("Trainer script:", TRAIN_SCRIPT)
print("Detected optional flags:")
for flag in ["--max-fallback-rate", "--strict-qualification-fallback", "--robust-rng-restore", "--reset-clf-optimizer-on-resume"]:
    print(f"  {flag}: {flag in SUPPORTED_FLAGS}")

Mounted at /content/drive
TMG_RESULTS_ROOT = /content/TMG-GAN/results


In [ ]:
# Cell 5: Validate dataset + prepare output paths
REQUIRED_DATA_FILES = ("x_train.csv", "y_train.csv", "x_test.csv", "y_test.csv")
missing = [name for name in REQUIRED_DATA_FILES if not (DATASET_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing dataset files in {DATASET_DIR}: {', '.join(missing)}")

OUT_DIR = RESULTS_ROOT / "outputs_tmg"
CACHE_DIR = RESULTS_ROOT / "data_cache"
CHECKPOINT_DIR = OUT_DIR / "checkpoints" / DATASET / RUN_NAME
METRICS_PATH = OUT_DIR / "metrics" / DATASET / f"{RUN_NAME}.json"
BASELINE_METRICS_PATH = OUT_DIR / "metrics" / DATASET / f"{BASELINE_RUN_NAME}.json"
CKPT_LATEST = CHECKPOINT_DIR / "latest.pt"
LOG_DIR = OUT_DIR / "logs"
LOG_MODE = "resume" if RESUME_MODE else "fresh"
LOG_FILE = LOG_DIR / f"{RUN_NAME}_{LOG_MODE}.log"

for d in (OUT_DIR, CACHE_DIR, CHECKPOINT_DIR, LOG_DIR, METRICS_PATH.parent):
    d.mkdir(parents=True, exist_ok=True)

if RESUME_MODE and (not CKPT_LATEST.exists()):
    raise FileNotFoundError(
        f"Resume checkpoint not found: {CKPT_LATEST}. Set TMG_RESUME=0 for fresh run or set TMG_RUN_NAME."
    )

print("Validation OK")
print("Dataset dir:", DATASET_DIR)
print("Checkpoint expected:", CKPT_LATEST)
print("Log file:", LOG_FILE)

train script: True -> /content/TMG-GAN/results/thesis_tmg_pipeline/scripts/train_tmg_gan.py
dataset dir: True -> /content/TMG-GAN/results/datasets/CICIDS2017
checkpoint: True -> /content/TMG-GAN/results/outputs_tmg/checkpoints/CICIDS2017/tmg_gan_tabular/latest.pt


In [ ]:
# Cell 6: Helpers for metrics and streamed logging
def read_best_f1(path: Path):
    if not path.exists():
        return None
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        return float(payload.get("best_f1"))
    except Exception:
        return None

def stream_command_with_log(cmd, log_path: Path, cwd: Path = None) -> None:
    print("Running:")
    print(" ".join(cmd))
    with open(log_path, "a", encoding="utf-8", buffering=1) as log_fp:
        log_fp.write("\n" + "=" * 80 + "\n")
        log_fp.write(f"[{datetime.now(timezone.utc).isoformat()}] START\n")
        log_fp.write(" ".join(cmd) + "\n")
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            cwd=str(cwd) if cwd is not None else None,
        )
        for line in process.stdout:
            print(line, end="")
            log_fp.write(line)
        process.wait()
        log_fp.write(f"[{datetime.now(timezone.utc).isoformat()}] END rc={process.returncode}\n")
    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, cmd)

def print_log_tail(path: Path, n: int = 40):
    if not path.exists():
        return
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n----- Log Tail -----")
    for line in lines[-n:]:
        print(line)
    print("----- End Log Tail -----")

comparison_metrics_path = METRICS_PATH if RESUME_MODE else BASELINE_METRICS_PATH
baseline_best_f1 = read_best_f1(comparison_metrics_path)
print("Comparison metrics:", comparison_metrics_path)
print("Baseline best_f1:", baseline_best_f1)

TMG_RESULTS_ROOT = /content/TMG-GAN/results


In [ ]:
# Cell 7: Install dependencies and run preflight
subprocess.run([PY, "-m", "pip", "install", "-r", str(REQUIREMENTS_FILE)], check=True, cwd=str(PROJECT_ROOT))

preflight_cmd = [
    PY,
    str(PREFLIGHT_SCRIPT),
    "--dataset",
    DATASET,
    "--data-root",
    str(DATA_ROOT),
    "--output-dir",
    str(OUT_DIR),
    "--cache-dir",
    str(CACHE_DIR),
    "--run-name",
    RUN_NAME,
]
if RESUME_MODE:
    preflight_cmd.append("--resume")

subprocess.run(preflight_cmd, check=True, cwd=str(PROJECT_ROOT))
print("Preflight OK")

train script: True -> /content/TMG-GAN/results/thesis_tmg_pipeline/scripts/train_tmg_gan.py
dataset dir: True -> /content/TMG-GAN/results/datasets/CICIDS2017
checkpoint: True -> /content/TMG-GAN/results/outputs_tmg/checkpoints/CICIDS2017/tmg_gan_tabular/latest.pt


In [ ]:
# Cell 8: Build train command with runtime-compatible optional flags
def add_optional_flag(cmd: list[str], flag: str, *values: str) -> bool:
    if flag in SUPPORTED_FLAGS:
        cmd.append(flag)
        cmd.extend(values)
        return True
    return False

AUG_TARGET_MODE = os.environ.get("TMG_AUG_TARGET_MODE", "second_max").strip() or "second_max"
MAX_SYN_MULTIPLIER = float(os.environ.get("TMG_MAX_SYN_MULT", "1.5"))
CLF_WEIGHT_MODE = os.environ.get("TMG_CLF_WEIGHTING", "none").strip() or "none"
STRICT_QUAL = os.environ.get("TMG_STRICT_QUAL", "1") == "1"
MAX_FALLBACK_RATE = float(os.environ.get("TMG_MAX_FALLBACK_RATE", "0.05"))

if RESUME_MODE:
    ckpt = torch.load(CKPT_LATEST, map_location="cpu", weights_only=False)
    phase = str(ckpt.get("phase", "gan"))
    gan_epoch = int(ckpt.get("gan_epoch", -1)) + 1
    clf_epoch = int(ckpt.get("clf_epoch", -1)) + 1

    gan_cfg = ckpt.get("gan_config", {})
    cfg = ckpt.get("config", {})
    resume_hidden_dim = int(
        gan_cfg.get(
            "gan_hidden_dim",
            cfg.get(
                "hidden_dim",
                ckpt["cd_model_state_dict"]["backbone.0.parametrizations.weight.original"].shape[0],
            ),
        )
    )
    resume_z_dim = int(
        gan_cfg.get(
            "z_dim",
            ckpt["generator_state_dicts"][0]["backbone.0.weight"].shape[1],
        )
    )

    target_gan_epochs = max(gan_epoch + 100, gan_epoch + 1) if phase == "gan" else max(1, gan_epoch)
    target_clf_epochs = max(clf_epoch + 100, clf_epoch + 1) if phase == "clf" else 200

    resume_args = ["--resume"]
    if not add_optional_flag(resume_args, "--robust-rng-restore"):
        print("Skipping unsupported optional flag: --robust-rng-restore")
    if not add_optional_flag(resume_args, "--reset-clf-optimizer-on-resume"):
        print("Skipping unsupported optional flag: --reset-clf-optimizer-on-resume")
else:
    resume_hidden_dim = int(os.environ.get("TMG_GAN_HIDDEN_DIM", "256"))
    resume_z_dim = int(os.environ.get("TMG_Z_DIM", "64"))
    target_gan_epochs = int(os.environ.get("TMG_GAN_EPOCHS", "300"))
    target_clf_epochs = int(os.environ.get("TMG_CLF_EPOCHS", "200"))
    resume_args = []

train_cmd = [
    PY,
    "-u",
    str(TRAIN_SCRIPT),
    "--dataset",
    DATASET,
    "--data-root",
    str(DATA_ROOT),
    "--output-dir",
    str(OUT_DIR),
    "--cache-dir",
    str(CACHE_DIR),
    "--run-name",
    RUN_NAME,
    "--gan-hidden-dim",
    str(resume_hidden_dim),
    "--z-dim",
    str(resume_z_dim),
    "--gan-epochs",
    str(target_gan_epochs),
    "--epochs",
    str(target_clf_epochs),
    "--gan-lr",
    "2e-4",
    "--cd-steps",
    "1",
    "--g-steps",
    "1",
    "--batch-size",
    "1024",
    "--gen-batch-size",
    "2048",
    "--hidden-warmup-epochs",
    "100",
    "--hidden-loss-weight",
    "1.0",
    "--gan-eval-interval",
    "10",
    "--augmentation-cap",
    "300000",
    "--augmentation-target-mode",
    AUG_TARGET_MODE,
    "--max-synthetic-multiplier",
    str(MAX_SYN_MULTIPLIER),
    "--max-rejects",
    "20",
    "--clf-class-weighting",
    CLF_WEIGHT_MODE,
    "--clf-effective-num-beta",
    "0.9999",
    "--clf-label-smoothing",
    "0.02",
    "--clf-lr-patience",
    "5",
    "--clf-lr-decay",
    "0.5",
    "--clf-min-lr",
    "3e-5",
    "--clf-early-stop-patience",
    "20",
    "--checkpoint-interval",
    "1",
    "--eval-interval",
    "1",
    *resume_args,
]

if not add_optional_flag(train_cmd, "--max-fallback-rate", str(MAX_FALLBACK_RATE)):
    print("Skipping unsupported optional flag: --max-fallback-rate")

if STRICT_QUAL and (not add_optional_flag(train_cmd, "--strict-qualification-fallback")):
    print("Skipping unsupported optional flag: --strict-qualification-fallback")

print("Train command ready.")

In [ ]:
# Cell 9: Run training
stream_command_with_log(train_cmd, LOG_FILE, cwd=PROJECT_ROOT)

Runtime cwd: /content/TMG-GAN/results/thesis_tmg_pipeline
Results-root candidates checked: 1
Results root: /content/TMG-GAN/results
Project root: /content/TMG-GAN/results/thesis_tmg_pipeline
Dataset dir: /content/TMG-GAN/results/datasets/CICIDS2017
Output dir: /content/TMG-GAN/results/outputs_tmg
Cache dir: /content/TMG-GAN/results/data_cache
Run mode: FRESH
Run name: tmg_gan_tabular_20260310_075419
Baseline run for comparison: tmg_gan_tabular
Checkpoint dir: /content/TMG-GAN/results/outputs_tmg/checkpoints/CICIDS2017/tmg_gan_tabular_20260310_075419


/tmp/ipykernel_645/2027399658.py:25: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_NAME = f"{RUN_STEM}_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"


In [ ]:
# Cell 10: Visualize metrics and print verdict
subprocess.run(
    [
        PY,
        str(VISUALIZE_SCRIPT),
        "--dataset",
        DATASET,
        "--output-dir",
        str(OUT_DIR),
        "--run-name",
        RUN_NAME,
    ],
    check=True,
    cwd=str(PROJECT_ROOT),
)

new_best_f1 = read_best_f1(METRICS_PATH)
print("New best_f1:", new_best_f1)
if baseline_best_f1 is None:
    print("Verdict: baseline missing; run completed and produced fresh metrics.")
elif new_best_f1 is None:
    print("Verdict: run completed but could not parse new best_f1.")
else:
    delta = new_best_f1 - baseline_best_f1
    if delta > 0:
        print(f"Verdict: IMPROVED (delta={delta:.6f}).")
    elif delta < 0:
        print(f"Verdict: NOT IMPROVED (delta={delta:.6f}).")
    else:
        print("Verdict: UNCHANGED (delta=0).")

print_log_tail(LOG_FILE, n=40)
print("Full log file:", LOG_FILE)

Validation OK
Checkpoint expected: /content/TMG-GAN/results/outputs_tmg/checkpoints/CICIDS2017/tmg_gan_tabular_20260310_075419/latest.pt
Log file: /content/TMG-GAN/results/outputs_tmg/logs/tmg_gan_tabular_20260310_075419_fresh.log


In [ ]:
# Cell 11: Daily checklist reminder
print("Daily checklist:")
print("1) Local: git add -A && git commit && git push")
print("2) Colab: run Cell 2 and Cell 3 to pull latest")
print("3) Verify Synced commit matches pushed commit")
print("4) Run Cells 4 to 10 in order")

Comparison metrics file: /content/TMG-GAN/results/outputs_tmg/metrics/CICIDS2017/tmg_gan_tabular.json
Baseline best_f1: None
